### MF baseline under the official protocol

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


In [ ]:
from src.data.implicit import load_movielens_ratings, build_user_history

raw = load_movielens_ratings(ROOT / "data/raw/ml-1m/ratings.dat")
train = pd.read_csv(PROCESSED / "train.csv")
validation = pd.read_csv(PROCESSED / "validation.csv")
validation_candidates = pd.read_csv(PROCESSED / "validation_candidates_100neg.csv")
users = pd.read_csv(PROCESSED / "user_mapping.csv")
movies = pd.read_csv(PROCESSED / "movie_mapping.csv")
user_map = dict(zip(users.user_id.astype(int), users.user_idx.astype(int)))
movie_map = dict(zip(movies.movie_id.astype(int), movies.movie_idx.astype(int)))
for frame in (train, validation):
    frame["user_idx"] = frame.user_id.map(user_map)
    frame["movie_idx"] = frame.movie_id.map(movie_map)
history = build_user_history(raw)
train_movies = sorted(train.movie_id.astype(int).unique())
print("train:", train.shape, "validation:", validation.shape)


In [ ]:
from src.models import MatrixFactorization
from src.training import fit_model

config = {"model": "MF", "embedding_dim": 32, "learning_rate": 0.001, "batch_size": 512, "epochs": 40, "patience": 10, "weight_decay": 1e-6, "train_negatives": 4, "seed": 42, "device": str(DEVICE)}
model = MatrixFactorization(len(users), len(movies), embedding_dim=32)
print(model)


In [ ]:
model, history_df = fit_model(
    model,
    train,
    history,
    train_movies,
    movie_map,
    validation,
    config,
    MODELS / "mf_best.pth",
    RESULTS / "mf_training_history.csv",
    optimizer_name="adam",
    validation_candidates=validation_candidates,
)
history_df.tail()